In [0]:
%pip install --upgrade "mlflow[databricks]>=3.1.0"
%restart_python

### Load config files and variables

In [0]:
import mlflow
import mlflow.genai.datasets
from pyspark.sql import functions as F
from databricks.connect import DatabricksSession
import pandas as pd
import math

catalog = "users"
schema = "alex_miller"
table = "wikipedia_source"
evaluation_table = "dspy_rag_eval_example"

# load MLflow config
config_path = "./config/config.yaml"
model_config = mlflow.models.ModelConfig(development_config=config_path)
mlflow_config = model_config.get("mlflow_config") or {}

# Ensure we talk to the Databricks tracking server for run + model logging
mlflow.set_tracking_uri("databricks")

# Set experiment based on config
user_name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
experiment_name = mlflow_config.get("experiment_name", "dspy_rag_agent_experiments")
experiment_info = mlflow.set_experiment(f"/Users/{user_name}/{experiment_name}")
print(f"MLflow Experiment info: {experiment_info}")

### Load the MLflow Eval Dataset

In [0]:
evaluation_uc_table_name = f"{catalog}.{schema}.{evaluation_table}"

# Load or create MLflow evaluation dataset
eval_dataset = mlflow.genai.datasets.get_dataset(
    name=evaluation_uc_table_name
)

display(eval_dataset.to_df())

### Run evaluation with predefined scores

In [0]:
from mlflow.deployments import get_deploy_client

endpoint_name = "agents_users-alex_miller-dspy_rag_agent"  # Replace with your serving endpoint's name
client = get_deploy_client("databricks")

def agent_predict_fn(**inputs):
    # The MLflow Deployments SDK expects a dict containing the inputs as per your model signature
    prediction = client.predict(endpoint=endpoint_name, inputs=inputs)
    # For most agent endpoints, the result will be a dict like {"response": <text>} or similar
    # You may need to adapt the returned structure depending on your application and agent
    return prediction

In [0]:
from mlflow.genai.scorers import (
    Correctness,
    RetrievalRelevance,
    RelevanceToQuery,
    Safety,
    Guidelines,
)

# Save the scorers as a variable so we can re-use them in step 7

predefined_scorers = [
        RetrievalGroundedness(),  # Checks if email content is grounded in retrieved data
        Guidelines(
            name="citation_generation",
            guidelines="The generated response must generate citations based on the user query",
        ),
        Guidelines(
            name="concise_communication",
            guidelines="The response MUST be concise and to the point. The response should communicate the key message efficiently without being overly brief or losing important context.",
        ),
        Guidelines(
            name="professional_tone",
            guidelines="The response must be in a professional tone.",
        ),
        RelevanceToQuery(),  # Checks if email addresses the user's request
        Safety(),  # Checks for harmful or inappropriate content
        RetrievalRelevance(),
        Correctness()
    ]

# Run evaluation with predefined scorers
eval_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=agent_predict_fn,
    scorers=predefined_scorers,
)

### View and interpret the results programmatically (or in the UI)

In [0]:
eval_traces = mlflow.search_traces(run_id=eval_results.run_id)

# eval_traces is a Pandas DataFrame that has the evaluated traces.  The column `assessments` includes each scorer's feedback.
display(eval_traces)

### Production monitoring
- Use predefined scorers or custom scorers, register them and re-use for online monitoring
- For more information see here: https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/production-monitoring

In [0]:
# from mlflow.genai.scorers import Safety, ScorerSamplingConfig

# # Register the scorer with a name and start monitoring
# safety_scorer = Safety().register(name="safety_scorer")  # name must be unique to experiment
# safety_scorer = safety_scorer.start(sampling_config=ScorerSamplingConfig(sample_rate=0.7))

Use guidelines-based LLM scorers

In [0]:
# from mlflow.genai.scorers import Guidelines, ScorerSamplingConfig

# # Create and register the guidelines scorer
# english_scorer = Guidelines(
#   name="english",
#   guidelines=["The response must be in English"]
# ).register(name="is_english")  # name must be unique to experiment

# # Start monitoring with the specified sample rate
# english_scorer = english_scorer.start(sampling_config=ScorerSamplingConfig(sample_rate=0.7))

User prompt-based scorers

In [0]:
# from mlflow.genai.scorers import scorer, ScorerSamplingConfig


# @scorer
# def formality(inputs, outputs, trace):
#     # Must be imported inline within the scorer function body
#     from mlflow.genai.judges.databricks import custom_prompt_judge
#     from mlflow.entities.assessment import DEFAULT_FEEDBACK_NAME

#     formality_prompt = """
#     You will look at the response and determine the formality of the response.

#     <request>{{request}}</request>
#     <response>{{response}}</response>

#     You must choose one of the following categories.

#     [[formal]]: The response is very formal.
#     [[semi_formal]]: The response is somewhat formal. The response is somewhat formal if the response mentions friendship, etc.
#     [[not_formal]]: The response is not formal.
#     """

#     my_prompt_judge = custom_prompt_judge(
#         name="formality",
#         prompt_template=formality_prompt,
#         numeric_values={
#             "formal": 1,
#             "semi_formal": 0.5,
#             "not_formal": 0,
#         },
#         model="databricks:/databricks-gpt-oss-20b",  # optional
#     )

#     result = my_prompt_judge(request=inputs, response=inputs)
#     if hasattr(result, "name"):
#         result.name = DEFAULT_FEEDBACK_NAME
#     return result

# # Register the custom scorer and start monitoring
# formality_scorer = formality.register(name="my_formality_scorer")  # name must be unique to experiment
# formality_scorer = formality_scorer.start(sampling_config=ScorerSamplingConfig(sample_rate=0.1))

Use custom scorer functions

In [0]:
# from mlflow.genai.scorers import scorer, ScorerSamplingConfig


# # Custom metric: Check if response mentions Databricks
# @scorer
# def mentions_databricks(outputs):
#     """Check if the response mentions Databricks"""
#     return "databricks" in str(outputs.get("response", "")).lower()

# # Custom metric: Response length check
# @scorer(aggregations=["mean", "min", "max"])
# def response_length(outputs):
#     """Measure response length in characters"""
#     return len(str(outputs.get("response", "")))

# # Custom metric with multiple inputs
# @scorer
# def response_relevance_score(inputs, outputs):
#     """Score relevance based on keyword matching"""
#     query = str(inputs.get("query", "")).lower()
#     response = str(outputs.get("response", "")).lower()

#     # Simple keyword matching (replace with your logic)
#     query_words = set(query.split())
#     response_words = set(response.split())

#     if not query_words:
#         return 0.0

#     overlap = len(query_words & response_words)
#     return overlap / len(query_words)

# # Register and start monitoring custom scorers
# databricks_scorer = mentions_databricks.register(name="databricks_mentions")
# databricks_scorer = databricks_scorer.start(sampling_config=ScorerSamplingConfig(sample_rate=0.5))

# length_scorer = response_length.register(name="response_length")
# length_scorer = length_scorer.start(sampling_config=ScorerSamplingConfig(sample_rate=1.0))

# relevance_scorer = response_relevance_score.register(name="response_relevance_score")  # name must be unique to experiment
# relevance_scorer = relevance_scorer.start(sampling_config=ScorerSamplingConfig(sample_rate=1.0))

Metric backfilling scorers for specific experiment

In [0]:
# from databricks.agents.scorers import backfill_scorers, BackfillScorerConfig

# job_id = backfill_scorers(
#     experiment_id="your-experiment-id",
#     scorers=[
#         BackfillScorerConfig(scorer=safety_scorer, sample_rate=0.8),
#         BackfillScorerConfig(scorer=response_length, sample_rate=0.9)
#     ],
#     start_time=datetime(2024, 1, 1),
#     end_time=datetime(2024, 1, 31)
# )